In [ ]:
!pip install -q pyarrow

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def find_input(filename, root="/kaggle/input"):
    matches = sorted(Path(root).rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one {filename} under {root}, found {len(matches)}: {matches}"
        )
    return matches[0]


HORIZON = 15
FEATURE_LAGS = [21, 28, 35]
ROLLING_WINDOWS = [7, 14, 28]

subset = pd.read_parquet(find_input("demandlens_subset.parquet"))
subset["date"] = pd.to_datetime(subset["date"])
subset = subset.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)
print(subset.shape)

activation_diag = pd.read_csv(
    find_input("series_activation_diagnostics.csv"), parse_dates=["activation_date"]
)

oil_by_date = subset.groupby("date")["dcoilwtico"].first().sort_index()
oil_lagged = oil_by_date.shift(HORIZON)

In [ ]:
subset = subset.merge(
    activation_diag[["store_nbr", "family", "activation_date"]],
    on=["store_nbr", "family"], how="left",
)
before = len(subset)
subset = subset[subset["date"] >= subset["activation_date"]].drop(columns=["activation_date"])
subset = subset.reset_index(drop=True)
print(f"Trimmed {before - len(subset)} pre-activation rows across all series")

In [ ]:
def classify_demand_pattern(series):
    nonzero = series[series > 0]
    n_periods = len(series)
    n_nonzero = len(nonzero)
    if n_nonzero == 0:
        return np.nan, np.nan, "no_demand"

    adi = n_periods / n_nonzero
    cv2 = (nonzero.std() / nonzero.mean()) ** 2 if nonzero.mean() > 0 else np.nan

    if adi < 1.32 and cv2 < 0.49:
        label = "smooth"
    elif adi >= 1.32 and cv2 < 0.49:
        label = "intermittent"
    elif adi < 1.32 and cv2 >= 0.49:
        label = "erratic"
    else:
        label = "lumpy"
    return adi, cv2, label


pattern_results = []
for (store_nbr, family), g in subset.groupby(["store_nbr", "family"]):
    s = g.sort_values("date")["sales"]
    adi, cv2, label = classify_demand_pattern(s)
    pattern_results.append({"store_nbr": store_nbr, "family": family, "adi": adi, "cv2": cv2, "pattern": label})

pattern_df = pd.DataFrame(pattern_results)
print(pattern_df.sort_values("pattern"))
print("\nPattern counts:\n", pattern_df["pattern"].value_counts())
pattern_df.to_csv("/kaggle/working/demand_pattern_classification.csv", index=False)

In [ ]:
subset["day_of_week"] = subset["date"].dt.dayofweek
subset["month"] = subset["date"].dt.month
subset["is_weekend"] = (subset["day_of_week"] >= 5).astype(int)
subset["day_of_month"] = subset["date"].dt.day
subset["week_of_year"] = subset["date"].dt.isocalendar().week.astype(int)

subset["dcoilwtico_lag"] = subset["date"].map(oil_lagged)
subset = subset.drop(columns=["dcoilwtico"])


def add_lag_rolling_features(df):
    df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)
    sales_by_series = df.groupby(["store_nbr", "family"])["sales"]

    for lag in FEATURE_LAGS:
        df[f"lag_{lag}"] = sales_by_series.shift(lag)

    for window in ROLLING_WINDOWS:
        min_periods = max(2, window // 2)
        df[f"rolling_mean_{window}"] = sales_by_series.transform(
            lambda s: s.shift(HORIZON).rolling(window, min_periods=min_periods).mean()
        )
        df[f"rolling_std_{window}"] = sales_by_series.transform(
            lambda s: s.shift(HORIZON).rolling(window, min_periods=min_periods).std()
        )
    return df


subset = add_lag_rolling_features(subset)
lag_cols = [f"lag_{lag}" for lag in FEATURE_LAGS]
print(subset[lag_cols + ["rolling_mean_7", "rolling_std_7"]].isna().sum())

In [ ]:
categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
for col in categorical_cols:
    subset[col] = subset[col].astype("category")

before = len(subset)
subset = subset.dropna(subset=[f"lag_{max(FEATURE_LAGS)}", "dcoilwtico_lag"]).reset_index(drop=True)
print(f"Dropped {before - len(subset)} rows lacking full lag history")

print(subset.isna().sum()[subset.isna().sum() > 0])
print(subset.dtypes)
print(subset["date"].min(), subset["date"].max())

subset.to_parquet("/kaggle/working/demandlens_features.parquet", index=False)
print("Saved: demandlens_features.parquet")
print(f"Final shape: {subset.shape}")